# LangChain

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **LangChain** — the framework for composing LLM calls, prompts, retrievers, tools, and memory into applications, built around **LCEL** (the LangChain Expression Language) and a single `Runnable` interface that everything implements.

## 1. What & Why

**LangChain** is an open-source framework (Python + JS/TS) for building applications on top of language models. Its job is to be the **standard interface and composition layer** between your app and the messy ecosystem around LLMs: dozens of model providers, vector stores, document loaders, embedding models, tools, and output formats — all behind one consistent API.

**The problem it solves: every LLM app re-invents the same plumbing.** You need to format a prompt from variables, call a model, parse the response into something usable, maybe retrieve context first (RAG), maybe let the model call tools, maybe remember the conversation. Hand-rolled, that's a pile of provider-specific glue that breaks when you swap GPT-4 for Claude or FAISS for pgvector. LangChain gives you **swappable, composable components** with a shared interface so you wire the *flow* once and change the *parts* freely.

**Two ideas worth holding onto:**
- **Everything is a `Runnable`.** Models, prompts, parsers, retrievers, and your own functions all implement the same interface (`invoke` / `batch` / `stream` / their async twins). Because they share an interface, you compose them with the pipe operator: `prompt | model | parser`. That's **LCEL**.
- **The package is deliberately small at the core.** `langchain-core` holds the abstractions (Runnable, messages, prompts, output parsers). Each integration (`langchain-openai`, `langchain-anthropic`, `langchain-chroma`, …) is a separate, independently-versioned package. You install only what you use.

**Reach for LangChain when:**
- You want **provider-agnostic** code — write the chain once, swap models/vector stores with a one-line change.
- You're building **RAG** or **tool-using** apps and want batteries-included loaders, splitters, retrievers, and output parsers rather than writing them yourself.
- You value the **huge integration surface** (hundreds of providers) and a standard streaming/batching/tracing story (LangSmith).

**Skip it (or stay minimal) when:**
- You make **one simple model call** with a couple of tools — the raw provider SDK is fewer moving parts.
- You need **complex, stateful, cyclic agent control flow** — that's now [[langgraph]]'s job (LangChain's old `AgentExecutor` is legacy). LangChain composes *linear* dataflow; LangGraph models *graphs* with state and cycles.
- You're sensitive to a **fast-moving API** and a deep dependency tree — both are real costs of the ecosystem.

## 2. Mental Model

Think of LangChain as **Unix pipes for LLM calls.**

In a shell you write `cat file | grep foo | sort` — small programs, each reading the previous one's output, joined by `|`. LCEL is the same idea for language-model work: each stage is a `Runnable`, and `|` feeds one stage's output into the next.

```
   input dict                                          final answer
   {"question": …}                                     "Paris."
        │                                                   ▲
        ▼                                                   │
   ┌──────────┐     ┌────────┐     ┌────────┐     ┌──────────────┐
   │ retriever│ ──▶ │ prompt │ ──▶ │ model  │ ──▶ │ output parser │
   │ + format │     │template│     │ (LLM)  │     │ (str / JSON)  │
   └──────────┘     └────────┘     └────────┘     └──────────────┘
        every box is a Runnable · joined by  |  · same invoke/batch/stream API
```

The whole pipeline is **itself a `Runnable`** — so you can `invoke` it on one input, `batch` it over a list, `stream` tokens as they arrive, or nest it inside a bigger chain. Composition is closed: chains of Runnables are Runnables.

One sentence: **components share one interface (`Runnable`), the `|` operator wires them into a pipeline, and the pipeline is just another component you can run, stream, or nest.**

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`Runnable`** | The universal interface every component implements: `invoke(x)` (one input), `batch(xs)` (many, parallelized), `stream(x)` (incremental output), plus async `ainvoke`/`abatch`/`astream`. Learn this once and everything composes. |
| **LCEL** | *LangChain Expression Language* — composing Runnables with `\|` (sequence) and dicts/`RunnableParallel` (branch/fan-out). Declarative dataflow; you get streaming, batching, and async for free. |
| **Chat model / LLM** | The model wrapper. `ChatOpenAI`, `ChatAnthropic`, etc. take/return **messages** (`SystemMessage`, `HumanMessage`, `AIMessage`). The modern default is chat models; raw-string `LLM`s are legacy. |
| **Prompt template** | `ChatPromptTemplate.from_messages([...])` turns input variables into a list of messages. Renders, doesn't call the model. |
| **Output parser** | Turns the model's raw text/message into a Python object: `StrOutputParser` (just the string), `JsonOutputParser`, `PydanticOutputParser` (validated schema). The tail of most chains. |
| **`RunnableLambda`** | Wraps any plain function as a Runnable so it can sit in a chain. `RunnablePassthrough` forwards input unchanged (often with `.assign(...)` to add keys). |
| **`RunnableParallel`** | Runs several Runnables on the **same input** and collects results into a dict — the standard way to build the `{"context": retriever, "question": passthrough}` shape for RAG. A bare `{...}` dict in a chain becomes one automatically. |
| **Retriever** | A Runnable that maps a query string → relevant `Document`s. Usually a **vector store** (`Chroma`, `FAISS`, `pgvector`) + an **embeddings** model, plus loaders and **text splitters** to build the index. |
| **Tools** | Functions the model can call (`@tool` decorator). Tool-calling is now driven through models' native function-calling; agent loops over tools live in [[langgraph]]. |
| **Memory / history** | Conversation state. The modern approach wraps a chain with `RunnableWithMessageHistory` (or uses LangGraph persistence) rather than the legacy `Memory` classes. |
| **LangSmith** | The hosted tracing/eval/observability product — set a couple of env vars and every chain run is traced. Optional but the reason many teams stay in the ecosystem. |

**The package layout matters.** `langchain-core` = the abstractions above. `langchain` = higher-level chains/helpers. `langchain-community` = community integrations. `langchain-openai` / `langchain-anthropic` / … = one provider each. Import models from their provider package, not from `langchain` itself.

## 4. Setup

LangChain is split into small packages — install the core plus only the integrations you need.

```bash
pip install langchain langchain-core          # core abstractions + helpers
pip install langchain-openai                  # one provider (or -anthropic, -google-genai, …)
pip install langchain-chroma chromadb         # a vector store, for RAG
pip install langsmith                         # optional: tracing/eval
```

To actually call a model you need that provider's key, e.g. `export OPENAI_API_KEY=...` (or `ANTHROPIC_API_KEY`). Tracing is opt-in via `LANGSMITH_TRACING=true` + `LANGSMITH_API_KEY`.

The first two worked examples below are **dependency-free**: they reimplement LCEL's `Runnable` + `|` mechanics in plain Python, so the notebook runs anywhere with no install or API key. The third cell shows the **real LangChain code** and runs it only if `langchain-openai` and an `OPENAI_API_KEY` are both present.

In [ ]:
# Installs are optional — examples 1 & 2 are pure stdlib and run offline.
# Uncomment to get the real framework used in example 3:
# %pip install langchain langchain-core langchain-openai

import importlib.util, os

lc_core = importlib.util.find_spec("langchain_core") is not None
lc_openai = importlib.util.find_spec("langchain_openai") is not None
has_key = bool(os.getenv("OPENAI_API_KEY"))
print("langchain-core installed:  ", lc_core)
print("langchain-openai installed:", lc_openai)
print("OPENAI_API_KEY present:    ", has_key)

## 5. Worked Examples

### Example 1 — `Runnable` + the `|` operator (LCEL from scratch)

LangChain's whole composition story is one interface plus one operator. Below we reimplement the essentials — `Runnable` with `invoke`/`batch`/`stream`, `RunnableLambda` to wrap a function, `RunnableSequence` for `|`, and `RunnableParallel` for fan-out — so you can see exactly what `prompt | model | parser` desugars to. This is a faithful (if tiny) mirror of `langchain_core.runnables`.

In [ ]:
from __future__ import annotations
from typing import Any, Callable, Iterator


class Runnable:
    """Minimal stand-in for langchain_core.runnables.Runnable."""

    def invoke(self, x: Any) -> Any:
        raise NotImplementedError

    def batch(self, xs: list) -> list:
        return [self.invoke(x) for x in xs]          # real LC parallelizes these

    def stream(self, x: Any) -> Iterator[Any]:
        yield self.invoke(x)                         # default: one chunk

    def __or__(self, other) -> "Runnable":
        # `a | b` -> compose. A plain function/dict is coerced to a Runnable.
        return RunnableSequence(self, _coerce(other))

    def __ror__(self, other) -> "Runnable":
        # `{...} | runnable` or `fn | runnable` -> coerce the left side first.
        return RunnableSequence(_coerce(other), self)


class RunnableLambda(Runnable):
    """Wrap any function so it can live in a chain."""

    def __init__(self, fn: Callable[[Any], Any]):
        self.fn = fn

    def invoke(self, x):
        return self.fn(x)


class RunnableSequence(Runnable):
    """`first | second`: feed first's output into second."""

    def __init__(self, first: Runnable, second: Runnable):
        self.first, self.second = first, second

    def invoke(self, x):
        return self.second.invoke(self.first.invoke(x))


class RunnableParallel(Runnable):
    """Run several Runnables on the SAME input -> dict of results (fan-out)."""

    def __init__(self, **steps):
        self.steps = {k: _coerce(v) for k, v in steps.items()}

    def invoke(self, x):
        return {k: r.invoke(x) for k, r in self.steps.items()}


def _coerce(obj) -> Runnable:
    if isinstance(obj, Runnable):
        return obj
    if isinstance(obj, dict):                        # a bare {...} becomes RunnableParallel
        return RunnableParallel(**obj)
    return RunnableLambda(obj)                        # a plain function


# --- Build a chain the LCEL way: prompt | model | parser ---------------------
prompt = RunnableLambda(lambda d: f"Question: {d['question']}")    # PromptTemplate
fake_model = RunnableLambda(lambda p: f"Echo[{p}]")                # the "LLM"
parser = RunnableLambda(lambda s: s.upper())                       # StrOutputParser

chain = prompt | fake_model | parser

print("invoke:", chain.invoke({"question": "what is LCEL?"}))
print("batch: ", chain.batch([{"question": "what is a retriever?"},
                              {"question": "name a parser"}]))
print("type:  ", type(chain).__name__, "(a chain is itself a Runnable)")

### Example 2 — A tiny RAG pipeline with `RunnableParallel`

The canonical RAG shape in LCEL is `{"context": retriever, "question": passthrough} | prompt | model | parser`. The `RunnableParallel` step runs the retriever **and** forwards the raw question on the same input, producing the dict the prompt needs. Here we plug a keyword-overlap retriever and a deterministic "model" into the primitives from Example 1 — no embeddings, no API — so the dataflow runs end to end with real output.

In [ ]:
# A miniature "vector store": documents + a keyword-overlap "retriever".
DOCS = [
    "LCEL composes Runnables with the | operator into a single pipeline.",
    "A retriever maps a query string to the most relevant documents.",
    "Output parsers turn a model's raw text into structured Python objects.",
    "langchain-core holds the abstractions; providers ship as separate packages.",
]

def _words(s: str) -> set:
    return {w.strip(".,").lower() for w in s.split()}

def retrieve(query: str, k: int = 2) -> list[str]:
    scored = sorted(DOCS, key=lambda d: len(_words(query) & _words(d)), reverse=True)
    return scored[:k]

def format_docs(docs: list[str]) -> str:
    return "\n".join(f"- {d}" for d in docs)

# A stand-in "model": extractive answer = the single most relevant retrieved line.
def fake_model(prompt: str) -> str:
    context = prompt.split("Context:\n", 1)[1].split("\nQuestion:", 1)[0]
    top = context.splitlines()[0].lstrip("- ")
    return f"Based on the context: {top}"

# RAG chain — note the {...} dict becomes a RunnableParallel automatically.
rag_chain = (
    {
        "context": RunnableLambda(lambda q: format_docs(retrieve(q))),
        "question": RunnableLambda(lambda q: q),          # passthrough
    }
    | RunnableLambda(
        lambda d: f"Answer using only the context.\n"
                  f"Context:\n{d['context']}\nQuestion: {d['question']}\nAnswer:"
    )
    | RunnableLambda(fake_model)
    | RunnableLambda(str.strip)                            # parser
)

q = "What does an output parser do?"
print("retrieved:", retrieve(q))
print("answer:   ", rag_chain.invoke(q))

### Example 3 — The real thing: `ChatPromptTemplate | ChatOpenAI | StrOutputParser`

This is the canonical LangChain chain with real classes. It calls a model, so it runs **only** if `langchain-openai` is installed *and* `OPENAI_API_KEY` is set; otherwise it prints the exact code you'd write. Notice it's the *same shape* as Examples 1–2 — `prompt | model | parser` — just with production components, and the chain still exposes `invoke` / `batch` / `stream`.

In [ ]:
SNIPPET = """
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a terse assistant. Answer in one sentence."),
    ("human", "{question}"),
])
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | model | StrOutputParser()

# Same Runnable interface as our toy version:
print(chain.invoke({"question": "What is LCEL in one line?"}))
print(chain.batch([{"question": "Name a vector store."},
                   {"question": "What is a retriever?"}]))
for chunk in chain.stream({"question": "List three LangChain packages."}):
    print(chunk, end="", flush=True)
"""

if lc_openai and os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a terse assistant. Answer in one sentence."),
        ("human", "{question}"),
    ])
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = prompt | model | StrOutputParser()
    print("Model:", chain.invoke({"question": "What is LCEL in one line?"}))
else:
    print("langchain-openai + OPENAI_API_KEY not both present — here is the real code:")
    print(SNIPPET)

## 6. Gotchas & Pitfalls

- **Import models from their provider package, not from `langchain`.** It's `from langchain_openai import ChatOpenAI`, not `from langchain import OpenAI`. The split into `langchain-core` + per-provider packages is intentional; mixing up imports (or pinning incompatible versions across packages) is the most common setup error.
- **Chat models vs. legacy `LLM`s.** Modern LangChain is message-based (`ChatOpenAI` takes/returns messages). The old string-in/string-out `LLM` classes still exist but are legacy — prefer chat models and `ChatPromptTemplate`.
- **`AgentExecutor` is legacy — use [[langgraph]] for agents.** The classic `initialize_agent` / `AgentExecutor` loop is deprecated. For tool-using, stateful, or cyclic agents, LangChain now points you to LangGraph (e.g. `create_react_agent`). LCEL is for *linear* dataflow; don't try to express loops/branches-with-state in raw LCEL.
- **A bare dict in a chain is a `RunnableParallel`.** `{"context": ..., "question": ...}` silently becomes a parallel fan-out. Handy, but if a value isn't a Runnable/function it won't behave as you expect — and forgetting the passthrough (`"question": RunnablePassthrough()`) is the classic RAG bug where the question never reaches the prompt.
- **`stream` only streams if every stage can.** A parser or lambda that needs the *whole* output (e.g. `JsonOutputParser` on incomplete JSON, or a function that calls `.upper()` on the full string) forces buffering. Token streaming is end-to-end only when each link is stream-friendly.
- **Output parsers fail on malformed output.** The model doesn't *have* to return valid JSON. `JsonOutputParser`/`PydanticOutputParser` raise on bad output — add format instructions to the prompt and consider `OutputFixingParser`/retries for production.
- **Fast-moving API + deep dependencies.** Imports and recommended patterns have churned across versions (`langchain` → `langchain-core` + integrations; memory → history → LangGraph). Pin versions, and check the docs' date — old tutorials rot fast.
- **Async/sync mirror methods.** Every Runnable has `invoke`/`batch`/`stream` *and* `ainvoke`/`abatch`/`astream`. In an async app, use the `a*` variants throughout; mixing sync calls into an event loop will block it.
- **It's a composition layer, not magic.** LangChain doesn't make a weak model smart or a bad retrieval good. Garbage context in → garbage answer out; the framework just makes wiring the pieces faster.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs LangChain |
|---|---|---|
| **LangChain (LCEL)** | Provider-agnostic chains; RAG and tool apps with batteries-included loaders/splitters/parsers; huge integration surface; LangSmith tracing | Fast-moving API, deep dependency tree; overkill for a single model call; *linear* dataflow only (agents now live in LangGraph) |
| **[[langgraph]]** | Stateful, cyclic, multi-step **agents** with checkpointing and human-in-the-loop | Lower-level graph wiring; more code. It's the *complement* to LangChain (often used together), not a drop-in for simple chains |
| **[[llamaindex-agents]]** (LlamaIndex) | **RAG-first** apps — rich indexing, query engines, and retrieval strategies out of the box | Narrower than LangChain's general composition; if you're mostly doing retrieval, often simpler |
| **[[haystack]]** | Production **pipeline** framework with a strong search/RAG heritage and typed components | Smaller integration ecosystem; different (node/pipeline) mental model |
| **Raw provider SDK** (`openai`, `anthropic`) | One or few model calls, full control, minimal deps | You hand-build prompt formatting, retries, streaming glue, RAG, and tool loops — exactly the plumbing LangChain removes |
| **DSPy** | **Optimizing** prompts/programs against metrics rather than hand-writing them | Different paradigm (compile-and-optimize), not a general app/integration layer |

**Rule of thumb:** for **one model call**, use the provider SDK. For **provider-agnostic chains and RAG** with lots of integrations, LangChain. For **stateful agent loops**, reach for LangGraph (with or alongside LangChain). For a **retrieval-heavy** app, weigh LlamaIndex. Many production stacks use **LangChain for the components + LangGraph for the control flow + LangSmith for observability** together.

Related notebooks: [[langgraph]], [[llamaindex-agents]], [[haystack]], [[crewai]], [[pydanticai]], [[react]].

## 8. Resources

- **Official docs** — https://python.langchain.com/docs/introduction/
- **LCEL / Runnable interface** — https://python.langchain.com/docs/concepts/runnables/
- **LCEL "why" + how-to** — https://python.langchain.com/docs/concepts/lcel/
- **Build a RAG app (tutorial)** — https://python.langchain.com/docs/tutorials/rag/
- **Conceptual guide (architecture & packages)** — https://python.langchain.com/docs/concepts/architecture/
- **API reference** — https://python.langchain.com/api_reference/
- **GitHub** — https://github.com/langchain-ai/langchain
- **LangSmith (tracing/eval)** — https://docs.smith.langchain.com/

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
class Runnable:
    ...


def coerce(obj):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE